# All-predictions map grid

Packaged from `build_all_predictions_grid.py`. Builds the one figure both
presentation notebooks below reuse: `figs_deck/all_predictions_grid.png`,
all twenty predicted-IMD rasters grouped by city on one colour scale. Run
this before the two deck notebooks if the figure needs regenerating.

In [ ]:
# -*- coding: utf-8 -*-
"""One figure, all twenty predicted-IMD rasters, grouped by city.

Milan (10, CLMS-trained then GHS-BUILT-S-trained) on the top two rows, Hanoi
(5) and HCMC (5) each on their own row below. Same colour scale and palette
as report/figs/fig_milan_raster_comparison.png (imd_cmap, 0-100%), so this
reads as a companion to that figure and to figs_deck/rmse.png.

Rasters are read decimated (rasterio out_shape) rather than at full
resolution -- this is a thumbnail grid, not a measurement figure.

    C:\\ProgramData\\anaconda3\\python.exe build_all_predictions_grid.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.enums import Resampling

MATEJ = Path(r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious")
REPO = Path(r"C:\Users\user\projects\IMD-Mapping")
OUT_PNG = REPO / "figs_deck" / "all_predictions_grid.png"

IMD_CMAP = ListedColormap(["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"])
THUMB = 260  # decimated read side, px

MILAN = [
    ("S2 stack",            MATEJ / "IMD/outputs_S2_stack/IMD_predicted_RF_S2_Milan.tif"),
    ("S2 percentile",       MATEJ / "IMD/outputs_S2_percentile_p10p25p50p75p90/IMD_predicted_RF_S2_Milan.tif"),
    ("emb_RF",              MATEJ / "IMD/outputs_v2/IMD_predicted_RF_spatialCV2_Milan.tif"),
    ("S2 median",           MATEJ / "IMD/outputs_S2_median/IMD_predicted_RF_S2_Milan.tif"),
    ("CLMS (target)",       MATEJ / "matej_files_codes/reference_IMD/IMD_2018_CLMS_Milan.tif"),
    ("S2 percentile\n\u00b7 GHSL", MATEJ / "IMD/outputs_S2_percentile_p10p25p50p75p90_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("S2 stack\n\u00b7 GHSL",      MATEJ / "IMD/outputs_S2_stack_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("emb_RF\n\u00b7 GHSL",        MATEJ / "IMD/outputs_GHSL/GHSL_predicted_RF_spatialCV2_Milan.tif"),
    ("S2 median\n\u00b7 GHSL",     MATEJ / "IMD/outputs_S2_median_GHSL/GHSL_predicted_RF_S2_Milan.tif"),
    ("GHS-BUILT-S\n(product)", REPO / "data/GHSL_2018_Milan_UTM32N.tif"),
]
HANOI_AOI = MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_localrf.tif"
HCMC_AOI = MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_localrf.tif"
HANOI = [
    ("emb_localrf",        MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_localrf.tif"),
    ("emb_zeroshot",       MATEJ / "IMD/outputs_transfer_v2/IMD_Hanoi_10m_zeroshot.tif"),
    ("s2_median_localrf",  MATEJ / "IMD/outputs_transfer_S2_median/IMD_Hanoi_10m_localrf_S2median.tif"),
    ("s2_median_zeroshot", MATEJ / "IMD/outputs_transfer_S2_median/IMD_Hanoi_10m_zeroshot_S2median.tif"),
    ("GHS-BUILT-S\n(target)", MATEJ / "matej_files_codes/reference_IMD/IMD_2018_Hanoi.tif", HANOI_AOI),
]
HCMC = [
    ("emb_localrf",        MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_localrf.tif"),
    ("emb_zeroshot",       MATEJ / "IMD/outputs_transfer_v2/IMD_HCMC_10m_zeroshot.tif"),
    ("s2_median_localrf",  MATEJ / "IMD/outputs_transfer_S2_median/IMD_HCMC_10m_localrf_S2median.tif"),
    ("s2_median_zeroshot", MATEJ / "IMD/outputs_transfer_S2_median/IMD_HCMC_10m_zeroshot_S2median.tif"),
    ("GHS-BUILT-S\n(target)", MATEJ / "matej_files_codes/reference_IMD/IMD_2018_HCMC.tif", HCMC_AOI),
]

ROWS = [("MILAN", MILAN[:5]), ("MILAN \u00b7 GHSL", MILAN[5:]),
        ("HANOI", HANOI + [None]),
        ("HCMC", HCMC + [None])]


def read_thumb(path, clip_to=None):
    """Decimated read of `path`, optionally windowed to `clip_to`'s bounds so
    a much-larger reference tile (the raw GHS-BUILT-S tiles for Hanoi/HCMC
    are ~10x the model AOI) is shown at the same extent as its row."""
    with rasterio.open(path) as src:
        if clip_to is not None:
            from rasterio.windows import from_bounds
            with rasterio.open(clip_to) as ref:
                b = ref.bounds
            win = from_bounds(*b, transform=src.transform)
            h, w = win.height, win.width
            scale = max(h, w) / THUMB
            out_h, out_w = max(1, int(h / scale)), max(1, int(w / scale))
            arr = src.read(1, window=win, out_shape=(1, out_h, out_w),
                            resampling=Resampling.average, boundless=True)
            extent = (b.left, b.right, b.bottom, b.top)
        else:
            h, w = src.height, src.width
            scale = max(h, w) / THUMB
            out_h, out_w = max(1, int(h / scale)), max(1, int(w / scale))
            arr = src.read(1, out_shape=(1, out_h, out_w), resampling=Resampling.average)
            b = src.bounds
            extent = (b.left, b.right, b.bottom, b.top)
        nodata = src.nodata
        arr = arr.astype("float32")
        if nodata is not None:
            arr[arr == nodata] = np.nan
        arr[(arr < 0) | (arr > 100)] = np.nan
    return arr, extent


fig, axes = plt.subplots(4, 5, figsize=(19, 15.5))
im_for_cbar = None
for r, (city_label, items) in enumerate(ROWS):
    for c in range(5):
        ax = axes[r, c]
        entry = items[c]
        if entry is None:
            ax.axis("off")
            continue
        name, path, clip_to = entry if len(entry) == 3 else (*entry, None)
        arr, extent = read_thumb(path, clip_to)
        im_for_cbar = ax.imshow(arr, extent=extent, cmap=IMD_CMAP, vmin=0, vmax=100)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(name, fontsize=10.5, pad=4)
        if c == 0:
            ax.text(-0.06, 0.5, city_label, transform=ax.transAxes, rotation=90,
                     ha="center", va="center", fontsize=12, fontweight="bold")
        print(f"  {city_label:14s} {name.replace(chr(10),' '):22s} ok")

fig.subplots_adjust(left=0.045, right=0.93, top=0.97, bottom=0.02,
                     wspace=0.08, hspace=0.28)
cax = fig.add_axes([0.945, 0.15, 0.014, 0.7])
fig.colorbar(im_for_cbar, cax=cax, label="Predicted IMD (%)")

fig.savefig(OUT_PNG, dpi=160, bbox_inches="tight")
plt.close(fig)
print("saved:", OUT_PNG)